In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
import numpy as np
import matplotlib.pyplot as plt
from src.gtsrb import NUM_CLASSES, GTSRB_CLASSES, get_dataloaders, save_predictions

In [2]:
train_augmentation = transforms.Compose([

    transforms.RandomRotation(25),

    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.25,
        saturation=0.2
    ),

    transforms.RandomResizedCrop(
        size=32
    ),

    transforms.ToTensor(),
    
    transforms.Normalize(
        mean=[0.3403, 0.3121, 0.3214],
        std=[0.2724, 0.2608, 0.2669]
    )

])

In [3]:
train_loader, val_loader, test_loader = get_dataloaders(img_size=32, batch_size=128, train_transform=train_augmentation)

In [4]:
def train_model(optimizer_name):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = nn.Sequential(

        nn.Conv2d(3, 32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Flatten(),

        nn.Linear(64 * 8 * 8, 128),
        nn.ReLU(),

        nn.Linear(128, NUM_CLASSES)

    ).to(device)

    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "SGD":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=0.001
        )

    elif optimizer_name == "Momentum":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=0.001,
            momentum=0.9
        )

    elif optimizer_name == "Adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=0.001
        )

    epochs = 5

    losses = []
    accuracies = []

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss

        losses.append(epoch_loss)

        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                predictions = outputs.argmax(dim=1)

                correct += (
                    predictions == labels
                ).sum().item()

                total += labels.size(0)

        accuracy = 100 * correct / total

        accuracies.append(accuracy)

        print(
            f"{optimizer_name} | "
            f"Epoch {epoch+1} | "
            f"Loss: {epoch_loss:.4f} | "
            f"Validation Accuracy: {accuracy:.2f}%"
        )

    return model, losses, accuracies

In [5]:
def metrics(model, test_loader, device):

    model.eval()

    cm = np.zeros((43, 43), dtype=int)

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predictions = torch.max(outputs, 1)

            for label, prediction in zip(labels, predictions):

                cm[label.item(), prediction.item()] += 1

    np.set_printoptions(threshold=np.inf)

    print("Matriz de Confusão:")
    print(cm)

    acc_class = [0 for _ in GTSRB_CLASSES]

    print("\nAcurácia por classe:")

    for i in range(43):

        total_class = np.sum(cm[i, :])

        if total_class > 0:

            acc = 100 * (cm[i, i] / total_class)

        else:

            acc = 0

        acc_class[i] = acc

        print(f"Classe {i}: {acc:.2f}%")

    worst_class = acc_class.index(min(acc_class))
    best_class = acc_class.index(max(acc_class))

    print(
        f"\nWorst class: {worst_class} "
        f"- {min(acc_class):.2f}%"
    )

    print(
        f"Best class: {best_class} "
        f"- {max(acc_class):.2f}%"
    )

    return cm, acc_class

In [ ]:
def generate_predictions(model):

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    model.eval()

    predictions_list = []

    with torch.no_grad():

        for images, _ in test_loader:

            images = images.to(device)

            outputs = model(images)

            predictions = outputs.argmax(dim=1)

            predictions_list.extend(
                predictions.cpu().numpy()
            )
        metrics(model, test_loader, device)
    return predictions_list

In [7]:
model_sgd, losses_sgd, accs_sgd = train_model("SGD")

c:\Users\rafae\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


SGD | Epoch 1 | Loss: 605.0922 | Validation Accuracy: 10.04%
SGD | Epoch 2 | Loss: 574.0526 | Validation Accuracy: 13.18%
SGD | Epoch 3 | Loss: 559.7707 | Validation Accuracy: 14.88%
SGD | Epoch 4 | Loss: 548.8456 | Validation Accuracy: 16.61%
SGD | Epoch 5 | Loss: 539.0981 | Validation Accuracy: 19.61%


In [8]:
model_momentum, losses_momentum, accs_momentum = train_model("Momentum")

Momentum | Epoch 1 | Loss: 552.7338 | Validation Accuracy: 25.53%
Momentum | Epoch 2 | Loss: 478.3730 | Validation Accuracy: 34.35%
Momentum | Epoch 3 | Loss: 432.9632 | Validation Accuracy: 41.70%
Momentum | Epoch 4 | Loss: 404.5984 | Validation Accuracy: 45.74%
Momentum | Epoch 5 | Loss: 378.6528 | Validation Accuracy: 53.36%


In [9]:
model_adam, losses_adam, accs_adam = train_model("Adam")

Adam | Epoch 1 | Loss: 470.4946 | Validation Accuracy: 46.42%
Adam | Epoch 2 | Loss: 358.4885 | Validation Accuracy: 61.00%
Adam | Epoch 3 | Loss: 314.6835 | Validation Accuracy: 64.17%
Adam | Epoch 4 | Loss: 287.8845 | Validation Accuracy: 68.90%
Adam | Epoch 5 | Loss: 273.7490 | Validation Accuracy: 75.51%


In [10]:
print('Metrics for sgd')
y_pred_sgd = generate_predictions(model_sgd)

Metrics for sgd


KeyboardInterrupt: 

In [ ]:
print('Metrics for momentum')
y_pred_momentum = generate_predictions(model_momentum)

In [ ]:
print('Metrics for adam')
y_pred_adam = generate_predictions(model_adam)


In [ ]:
epochs_range = range(1, 6)

plt.figure(figsize=(8,5))

plt.plot(
    epochs_range,
    losses_sgd,
    marker='o',
    label='Loss'
)

plt.title('Curva de Loss')
plt.xlabel('Épocas')
plt.ylabel('Loss')

plt.xticks(epochs_range)

plt.grid(True)

plt.legend()

plt.show()


plt.figure(figsize=(8,5))

plt.plot(
    epochs_range,
    accs_sgd,
    marker='o',
    label='Validation Accuracy'
)

plt.title('Curva de Accuracy')
plt.xlabel('Épocas')
plt.ylabel('Accuracy (%)')

plt.xticks(epochs_range)

plt.grid(True)

plt.legend()

plt.show()

In [ ]:
epochs_range = range(1, 6)

plt.figure(figsize=(8,5))

plt.plot(
    epochs_range,
    losses_momentum,
    marker='o',
    label='Loss'
)

plt.title('Curva de Loss')
plt.xlabel('Épocas')
plt.ylabel('Loss')

plt.xticks(epochs_range)

plt.grid(True)

plt.legend()

plt.show()


plt.figure(figsize=(8,5))

plt.plot(
    epochs_range,
    accs_momentum,
    marker='o',
    label='Validation Accuracy'
)

plt.title('Curva de Accuracy')
plt.xlabel('Épocas')
plt.ylabel('Accuracy (%)')

plt.xticks(epochs_range)

plt.grid(True)

plt.legend()

plt.show()

In [ ]:
epochs_range = range(1, 6)

plt.figure(figsize=(8,5))

plt.plot(
    epochs_range,
    losses_adam,
    marker='o',
    label='Loss'
)

plt.title('Curva de Loss')
plt.xlabel('Épocas')
plt.ylabel('Loss')

plt.xticks(epochs_range)

plt.grid(True)

plt.legend()

plt.show()


plt.figure(figsize=(8,5))

plt.plot(
    epochs_range,
    accs_adam,
    marker='o',
    label='Validation Accuracy'
)

plt.title('Curva de Accuracy')
plt.xlabel('Épocas')
plt.ylabel('Accuracy (%)')

plt.xticks(epochs_range)

plt.grid(True)

plt.legend()

plt.show()